# Validation: NRTL activity coefficients vs thermo

Compare chemthermo NRTL activity coefficients against `thermo`'s NRTL helper for a binary
mixture. Results are printed as compact tables and a final assertions cell enforces the same
tolerances as `tests/validation/test_flash_vs_thermo.py`.


In [ ]:
from __future__ import annotations

from pathlib import Path
import sys
from typing import Any

import numpy as np
import chemthermo as ct

try:
    import thermo
except ModuleNotFoundError as exc:
    raise ModuleNotFoundError(
        "This notebook requires the 'thermo' package. Install with: pip install thermo"
    ) from exc


def _ensure_repo_root_on_path() -> None:
    cwd = Path.cwd()
    for candidate in (cwd, *cwd.parents):
        if (candidate / "pyproject.toml").exists():
            if str(candidate) not in sys.path:
                sys.path.insert(0, str(candidate))
            return


_ensure_repo_root_on_path()

from notebooks._nb_utils import print_table

NRTL_gammas = thermo.nrtl.NRTL_gammas

# Tolerances aligned with tests/validation/test_flash_vs_thermo.py
GAMMA_TOL = {"rel": 2e-3, "abs": 2e-3}


In [ ]:
case = {
    "name": "methane_ethane_240K",
    "components": ["Methane", "Ethane"],
    "zs": [0.50, 0.50],
    "temperature_K": 240.0,
}


In [ ]:
components = case["components"]
zs = case["zs"]
temperature_K = case["temperature_K"]

mixture = ct.Mixture.from_database(components, zs, normalize=True)
model = ct.NRTL(parameters=ct.ActivityParameters.load("NRTL"))
gammas = model.activity_coefficients(
    mixture=mixture,
    temperature_K=temperature_K,
    composition=mixture.fractions,
)

params = ct.NRTLParameters.load()
tau, alpha = params.for_components(components)
ref_gammas = NRTL_gammas(xs=list(mixture.fractions), taus=tau, alphas=alpha)

print_table(
    [
        {
            "case": case["name"],
            "components": components,
            "T [K]": temperature_K,
            "z": zs,
        }
    ],
    title="Input summary",
)

print_table(
    [
        {"source": "chemthermo", "gamma": gammas},
        {"source": "thermo", "gamma": ref_gammas},
    ],
    title="Results",
)

rows = []
for comp, g, rg in zip(components, gammas, ref_gammas):
    diff = float(g) - float(rg)
    rel = diff / (abs(float(rg)) if abs(float(rg)) > 1e-12 else 1.0)
    rows.append({"component": comp, "chem_gamma": g, "thermo_gamma": rg, "abs_diff": diff, "rel_diff": rel})
print_table(rows, title="Differences")


In [ ]:
# Assertions (mirrors tests/validation/test_flash_vs_thermo.py)
assert np.allclose(gammas, ref_gammas, rtol=GAMMA_TOL["rel"], atol=GAMMA_TOL["abs"])
